In [1]:
# ====================================================
# 🔧 STEP 1: Mount Google Drive
# ====================================================
from google.colab import drive
drive.mount('/content/drive')

# Create a working directory inside Drive
import os
WORK_DIR = '/content/drive/MyDrive/numeric_finetune_data'
os.makedirs(WORK_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
#!pip install -q transformers datasets peft accelerate wandb

In [2]:
# =========================================================
# ModernBERT + LoRA + Triplet Contrastive Training
# =========================================================

import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
)

from peft import LoraConfig, get_peft_model
from accelerate import Accelerator
import wandb
from tqdm import tqdm

In [3]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


False

In [4]:
!ls drive/MyDrive/numeric_finetune_data/NumerSense

data			      test_same_extracted.jsonl
happy-transformer	      test_same.jsonl
LICENSE			      test_sub_extracted.jsonl
Numeracy_600K_comment.json    test_sub.jsonl
README.md		      train_extracted_improved.jsonl
results			      train_extracted_improved_nodup.jsonl
src			      train_extracted.jsonl
test_extracted.jsonl	      train.jsonl
test_generic_extracted.jsonl  val_extracted.jsonl
test_generic.jsonl	      val.jsonl
test.jsonl


In [5]:

! head -2 drive/MyDrive/numeric_finetune_data/NumerSense/train_extracted.jsonl


{"anchor": "$27.9M City of Middletown, Connecticut Citigroup Global Markets Inc", "positive": "$29.61M City of Middletown, Connecticut Citigroup Global Markets Inc", "negative": "$23.6M City of Middletown, Connecticut Citigroup Global Markets Inc", "number": "27.9", "positive_rewritten": "Citigroup Global Markets Inc in Middletown, Connecticut with a value of $29.61M", "negative_rewritten": "Citigroup Global Markets Inc in Middletown, Connecticut with a value of $23.6M", "positive_number": "29.61", "negative_number": 23.6}
{"anchor": "Ex-N.Y. Senate leader Bruno asks state for $2.4 million in legal fees", "positive": "Ex-N.Y. Senate leader Bruno asks state for $2.07 million in legal fees", "negative": "Ex-N.Y. Senate leader Bruno asks state for $3.36 million in legal fees", "number": "2.4", "positive_rewritten": "Former New York Senate leader Bruno requests $2.07 million from the state for legal expenses", "negative_rewritten": "Former New York Senate leader Bruno seeks $3.36 million f

In [6]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "answerdotai/ModernBERT-base"

LORA_MODEL = "drive/MyDrive/numeric_finetune_data/trained_model/modernbert_lora_contrastive_final"

TRAIN_FILE = "drive/MyDrive/numeric_finetune_data/NumerSense/train_extracted_improved_nodup.jsonl"     # ~79k
VAL_FILE   = "drive/MyDrive/numeric_finetune_data/NumerSense/val_extracted.jsonl"       # ~10k

MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
MARGIN = 0.2

WANDB_PROJECT = "modernbert-numeracy-lora-corrected-final"

# Checkpointing settings
CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints_final_margin")
RESUME_FROM_CHECKPOINT = False # Set to True to resume training
SAVE_CHECKPOINT_STEPS = 500 # Save a checkpoint every N steps

In [7]:
# =========================================================
# DATASET
# =========================================================

class TripletDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.data = []
        with open(path) as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor": self.tokenizer(
                item["anchor"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "positive": self.tokenizer(
                item["positive_rewritten"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "negative": self.tokenizer(
                item["negative_rewritten"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "positive_number": float(item["positive_number"]),
            "negative_number": float(item["negative_number"]),
            "anchor_number": float(item["number"]),
        }

In [8]:
# =========================================================
# COLLATOR (DataCollatorWithPadding for triplets)
# =========================================================

def make_triplet_collator(tokenizer):
    base_collator = DataCollatorWithPadding(tokenizer)

    def collate(batch):
        return {
            "anchor": base_collator([b["anchor"] for b in batch]),
            "positive": base_collator([b["positive"] for b in batch]),
            "negative": base_collator([b["negative"] for b in batch]),
            "positive_number": torch.tensor([b["positive_number"] for b in batch], dtype=torch.float),
            "negative_number": torch.tensor([b["negative_number"] for b in batch], dtype=torch.float),
            "anchor_number": torch.tensor([b["anchor_number"] for b in batch], dtype=torch.float),
        }

    return collate

In [9]:
# =========================================================
# MEAN POOLING (IMPORTANT FOR MODERNBERT)
# =========================================================

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

In [10]:
# =========================================================
# MODEL WRAPPER
# =========================================================

class ContrastiveModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        hidden_size = encoder.config.hidden_size
        self.numeric_head = nn.Linear(hidden_size, 1)  # sees raw emb
        # Optional: separate projection for cosine space
        self.metric_proj = nn.Linear(hidden_size, hidden_size)  # sees raw, outputs normalized

    def encode(self, batch_part):
        out = self.encoder(
            input_ids=batch_part["input_ids"],
            attention_mask=batch_part["attention_mask"],
        )
        return mean_pooling(out, batch_part["attention_mask"])  # raw

    def forward(self, batch):
        a_emb = self.encode(batch["anchor"])
        p_emb = self.encode(batch["positive"])
        n_emb = self.encode(batch["negative"])

        # Head sees raw — preserves magnitude signal
        a_score = self.numeric_head(a_emb).squeeze(-1)
        p_score = self.numeric_head(p_emb).squeeze(-1)
        n_score = self.numeric_head(n_emb).squeeze(-1)

        # Cosine loss sees projected + normalized — clean directional space
        a_proj = F.normalize(self.metric_proj(a_emb), dim=-1)
        p_proj = F.normalize(self.metric_proj(p_emb), dim=-1)
        n_proj = F.normalize(self.metric_proj(n_emb), dim=-1)

        return {
            "a_emb": a_proj,    # for triplet + log-distance loss
            "p_emb": p_proj,
            "n_emb": n_proj,
            "a_score": a_score, # for head + rank loss
            "p_score": p_score,
            "n_score": n_score,
        }


In [11]:
import torch
import torch.nn.functional as F


def improved_numeric_loss(
    anchor_emb,
    pos_emb,
    neg_emb,
    anchor_score,       # pre-computed by numeric_head(raw_emb) in model.forward()
    pos_score,
    neg_score,
    anchor_value,       # ground truth numeric values, strictly > 0
    pos_value,
    neg_value,
    base_margin=0.2,
    alpha=0.5,          # metric_loss vs supervision_loss balance
    beta=0.6,           # within supervision: head_loss vs rank_loss balance
    eps=1e-8,
):
    """
    Multi-objective loss to teach ModernBERT numerical ordering via LoRA.

    Loss structure:
        total_loss = alpha * metric_loss + (1 - alpha) * supervision_loss

        where:
            metric_loss     = triplet_loss + log_distance_loss   (cosine space)
            supervision_loss = beta * head_loss + (1-beta) * rank_loss  (head space)

    Hyperparameters:
        alpha : float in (0, 1)
            Controls the balance between metric learning (cosine space) and
            direct supervision (head space).
            alpha → 1.0  : model focuses on getting cosine distances right
            alpha → 0.0  : model focuses on the numeric head predictions
            Recommended starting point: 0.5

        beta : float in (0, 1)
            Within the supervision side, controls absolute regression vs
            relative ordering.
            beta  → 1.0  : emphasize absolute log-value regression
            beta  → 0.0  : emphasize monotonic rank ordering
            Recommended starting point: 0.6 (slight preference for regression
            since it provides a stronger absolute anchor signal)

    Args:
        anchor/pos/neg_emb   : (B, H) raw or metric_proj embeddings from model.forward()
                               normalization is handled internally
        anchor/pos/neg_score : (B,)   numeric_head predictions on RAW embeddings
        anchor/pos/neg_value : (B,)   ground truth numeric values, strictly > 0
        base_margin          : base margin for dynamic triplet loss
        alpha                : see above
        beta                 : see above
        eps                  : numerical stability for log

    Returns:
        total_loss : scalar
        components : dict of individual loss values for logging
    """


    # ------------------------------------------------------------------
    # 1. Normalize embeddings for cosine space only
    #    Scores were computed on RAW embs in model.forward() so
    #    normalization here has no effect on head_loss or rank_loss.
    # ------------------------------------------------------------------
    anchor = F.normalize(anchor_emb, dim=-1)
    pos    = F.normalize(pos_emb,    dim=-1)
    neg    = F.normalize(neg_emb,    dim=-1)

    # ------------------------------------------------------------------
    # 2. Cosine distances  ∈ [0, 2]
    # ------------------------------------------------------------------
    pos_cos_dist = 1.0 - F.cosine_similarity(anchor, pos, dim=-1)  # (B,)
    neg_cos_dist = 1.0 - F.cosine_similarity(anchor, neg, dim=-1)  # (B,)

    # ------------------------------------------------------------------
    # 3. Log-space numeric distances  ∈ [0, ∞)
    # ------------------------------------------------------------------
    log_a = torch.log1p(anchor_value )   # (B,)
    log_p = torch.log1p(pos_value    )
    log_n = torch.log1p(neg_value    )

    log_pos_dist = torch.abs(log_a - log_p)   # (B,)
    log_neg_dist = torch.abs(log_a - log_n)   # (B,)

    # ------------------------------------------------------------------
    # 4. Dynamic Triplet Loss                         [metric space]
    #
    #    Uses cosine DISTANCE — penalizes when anchor is farther from
    #    positive than from negative.
    #
    #    Dynamic margin scales with the log-space numeric gap:
    #      large neg gap but small pos gap → harder penalty
    #      margin ∈ [base_margin, 2 * base_margin] via sigmoid
    # ------------------------------------------------------------------
    log_diff   = log_neg_dist - log_pos_dist
    dyn_margin = base_margin * (1.0 + torch.sigmoid(log_diff))   # (B,)

    triplet_loss = F.relu(pos_cos_dist - neg_cos_dist + dyn_margin).mean()

    # ------------------------------------------------------------------
    # 5. Log-Space Distance Alignment                 [metric space]
    #
    #    Aligns cosine distance magnitude with numeric log-ratio distance.
    #    Numbers are perceived on a log scale: 1→10 ~ 10→100.
    #
    #    Scale safety:
    #      cosine_dist / 2        maps [0, 2]  → [0, 1]
    #      tanh(log_dist)         maps [0, ∞)  → [0, 1)   (saturates gracefully)
    #    Both sides in [0, 1] → MSE is well-behaved, no blow-up on large gaps.
    # ------------------------------------------------------------------
    norm_pos_cos = pos_cos_dist / 2.0
    norm_neg_cos = neg_cos_dist / 2.0

    target_pos = torch.tanh(log_pos_dist)
    target_neg = torch.tanh(log_neg_dist)

    log_distance_loss = (
        F.mse_loss(norm_pos_cos, target_pos) +
        F.mse_loss(norm_neg_cos, target_neg)
    )

    # ------------------------------------------------------------------
    # 6. Head Regression Loss                         [head space]
    #
    #    numeric_head predicts log(value) from RAW embeddings, preserving
    #    magnitude signal. Scores are pre-computed in model.forward().
    #
    #    Divided by 3 to average over anchor/pos/neg roles so that
    #    lambda_head is on the same scale as other loss terms.
    # ------------------------------------------------------------------
    head_loss = (
        F.mse_loss(anchor_score, log_a) +
        F.mse_loss(pos_score,    log_p) +
        F.mse_loss(neg_score,    log_n)
    ) / 3.0

    # ------------------------------------------------------------------
    # 7. Monotonic Ranking Loss                       [head space]
    #
    #    Enforces: if value_a > value_b → pred_a > pred_b.
    #    Soft margin ranking: penalizes inversions and near-ties.
    #
    #    Complements head_loss:
    #      head_loss  → absolute accuracy  (where on the number line)
    #      rank_loss  → relative ordering  (which one is larger)
    # ------------------------------------------------------------------
    margin_rank = 0.1

    ap_sign = torch.sign(log_a - log_p)   # (B,)  +1 if anchor > pos
    an_sign = torch.sign(log_a - log_n)   # (B,)  +1 if anchor > neg

    rank_loss = (
        F.relu(margin_rank - ap_sign * (anchor_score - pos_score)).mean() +
        F.relu(margin_rank - an_sign * (anchor_score - neg_score)).mean()
    ) / 2.0

    # ------------------------------------------------------------------
    # 8. Grouped Weighted Loss
    #
    #    metric_loss     : cosine space — triplet + log_distance
    #    supervision_loss: head  space  — regression + ranking
    #
    #    total = alpha * metric + (1 - alpha) * supervision
    #
    #    Within supervision:
    #    supervision = beta * head + (1 - beta) * rank
    # ------------------------------------------------------------------
    metric_loss      = triplet_loss + log_distance_loss
    supervision_loss = beta * head_loss + (1.0 - beta) * rank_loss

    total_loss = alpha * metric_loss + (1.0 - alpha) * supervision_loss

    components = {
        "triplet":      triplet_loss.item(),
        "log_dist":     log_distance_loss.item(),
        "head":         head_loss.item(),
        "rank":         rank_loss.item(),
        "metric":       metric_loss.item(),
        "supervision":  supervision_loss.item(),
        "total":        total_loss.item(),
    }

    return total_loss, components




In [12]:
del base_model
del model


NameError: name 'base_model' is not defined

In [13]:
accelerator = Accelerator()
#wandb.init(project=WANDB_PROJECT)

# Tokenizer & Base Model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)



# -----------------------------------------------------
# LoRA CONFIG (attention layers only)
# -----------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=[
        "Wqkv",       # attention: combined Q/K/V projection
        "out_proj",   # attention: output projection
        "Wi",         # FFN: gated input projection (GLU gate + up proj combined)
        "Wo",         # FFN: down projection
    ],
    bias="none",
    task_type="FEATURE_EXTRACTION",
)

base_model = get_peft_model(base_model, lora_config)
model = ContrastiveModel(base_model)
"""
# LORA loading
# Load base architecture
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)

# Attach LoRA weights
base_model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL,
    is_trainable=True
)

model = ContrastiveModel(base_model)
"""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'\n# LORA loading\n# Load base architecture\nfrom peft import PeftModel\n\ntokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\nbase_model = AutoModel.from_pretrained(MODEL_NAME)\n\n# Attach LoRA weights\nbase_model = PeftModel.from_pretrained(\n    base_model,\n    LORA_MODEL,\n    is_trainable=True\n)\n\nmodel = ContrastiveModel(base_model)\n'

In [14]:
#model.encoder.print_trainable_parameters()


In [15]:
# -----------------------------------------------------
# Data
# -----------------------------------------------------
train_dataset = TripletDataset(TRAIN_FILE, tokenizer)
val_dataset   = TripletDataset(VAL_FILE, tokenizer)

collate_fn = make_triplet_collator(tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [16]:
counter = 0
for i, item in enumerate(train_dataset.data):
    a = float(item["number"])
    p = float(item["positive_number"])
    n = float(item["negative_number"])

    if a <= 0 or p <= 0 or n <= 0:
        counter +=1
        #print(f"idx={i}  anchor={a}  pos={p}  neg={n}")
print(counter)

4444


In [17]:
batch = next(iter(train_loader))
batch

{'anchor': {'input_ids': tensor([[50281,  2775, 30726,  ..., 34916,  6028, 50282],
         [50281,  5648,    41,  ..., 50283, 50283, 50283],
         [50281,  3338,  7142,  ..., 50283, 50283, 50283],
         ...,
         [50281,    47,  2354,  ..., 50283, 50283, 50283],
         [50281, 25595, 21346,  ..., 50283, 50283, 50283],
         [50281,  2775, 30726,  ..., 50283, 50283, 50283]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]])},
 'positive': {'input_ids': tensor([[50281,   510, 17340,  ..., 50283, 50283, 50283],
         [50281,    41,     7,  ..., 50283, 50283, 50283],
         [50281,  3338,  7142,  ..., 50283, 50283, 50283],
         ...,
         [50281,    47,  2354,  ..., 50283, 50283, 50283],
         [50281,   510, 23585,  ..., 50283, 50283, 50283],
         [50281,   510,  81

In [18]:
RESUME_FROM_CHECKPOINT = True
RESUME_FROM_CHECKPOINT

True

In [19]:
# Split parameters into three groups
lora_params = [
    p for n, p in model.named_parameters()
    if "lora_" in n and p.requires_grad
]
head_params = list(model.numeric_head.parameters())

# If using metric_proj:
proj_params = list(model.metric_proj.parameters())

optimizer = torch.optim.AdamW([
    {"params": lora_params,  "lr": 2e-4,  "weight_decay": 0.01},
    {"params": head_params,  "lr": 1e-3,  "weight_decay": 0.0},
   {"params": proj_params, "lr": 2e-4,  "weight_decay": 0.01},
], betas=(0.9, 0.999), eps=1e-8)

In [ ]:
#proj_params

In [20]:


model, optimizer, train_loader, val_loader = accelerator.prepare(
    model, optimizer, train_loader, val_loader
)

# =========================================================
# CHECKPOINTING: Resume from Checkpoint (moved here after prepare)
# =========================================================
start_epoch = 0
global_step = 0

# Ensure the checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if RESUME_FROM_CHECKPOINT:
    # Check for a specific file to confirm checkpoint existence
    # Accelerator saves a 'pytorch_model.bin' and 'optimizer.bin' along with other states.
    if os.path.exists(os.path.join(CHECKPOINT_DIR, "model.safetensors")):
        accelerator.load_state(CHECKPOINT_DIR)
        accelerator.print(f"Resuming training from checkpoint in {CHECKPOINT_DIR}")

        # Load metadata (epoch and global_step) if available
        metadata_path = os.path.join(CHECKPOINT_DIR, "training_metadata.json")
        if accelerator.is_main_process and os.path.exists(metadata_path):
            with open(metadata_path, 'r') as f:
                metadata = json.load(f)
                start_epoch = metadata.get("epoch", 0)
                global_step = metadata.get("global_step", 0)
            accelerator.print(f"Resumed epoch: {start_epoch}, global_step: {global_step}")
        elif not accelerator.is_main_process:
            # All processes need to wait for the main process to load metadata
            accelerator.wait_for_everyone()
            if os.path.exists(metadata_path):
                 with open(metadata_path, 'r') as f:
                    metadata = json.load(f)
                    start_epoch = metadata.get("epoch", 0)
                    global_step = metadata.get("global_step", 0)
    else:
        accelerator.print(f"No checkpoint found at {CHECKPOINT_DIR}. Starting fresh.")
else:
    accelerator.print("Starting training from scratch (RESUME_FROM_CHECKPOINT is False).")

accelerator.print(f"Initial epoch: {start_epoch}, initial global_step: {global_step}")

Resuming training from checkpoint in /content/drive/MyDrive/numeric_finetune_data/checkpoints_final_margin
Resumed epoch: 3, global_step: 8500
Initial epoch: 3, initial global_step: 8500


In [21]:
start_epoch,global_step

(3, 8500)

In [22]:
wandb.init(project=WANDB_PROJECT) # Initialize wandb with project name

# Ensure the checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for epoch in range(start_epoch, EPOCHS+3): # Start from 'start_epoch'
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        disable=not accelerator.is_main_process,
        desc=f"Epoch {epoch+1}"
    )

    for step, batch in enumerate(progress_bar):
        # if epoch == 2 and step < 7500:
        #     continue
        # Calculate current_global_step, accounting for resumed training
        current_global_step = global_step + (epoch - start_epoch) * len(train_loader) + step
        #print(batch)
        outputs = model(batch)
        loss, components = improved_numeric_loss(
                            anchor_emb   = outputs["a_emb"],
                            pos_emb      = outputs["p_emb"],
                            neg_emb      = outputs["n_emb"],
                            anchor_score = outputs["a_score"],
                            pos_score    = outputs["p_score"],
                            neg_score    = outputs["n_score"],
                            anchor_value = batch["anchor_number"],
                            pos_value    = batch["positive_number"],
                            neg_value    = batch["negative_number"],
                            alpha        = 0.5,
                            beta         = 0.4,
                        )

        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        # Log every 100 steps for WandB or if it's the very first step
        if (current_global_step + 1) % 100 == 0 or current_global_step == 0:
            accelerator.print(f"Step {current_global_step+1} | Loss {loss.item():.4f}")
            wandb.log({
                "train_loss_step": loss.item(),
                "global_step": current_global_step + 1,
                "epoch": epoch
            })

        # Checkpoint saving logic every SAVE_CHECKPOINT_STEPS
        if (current_global_step + 1) % SAVE_CHECKPOINT_STEPS == 0:
            accelerator.save_state(CHECKPOINT_DIR)
            # Save metadata (epoch, global_step) alongside the model
            if accelerator.is_main_process:
                metadata = {"epoch": epoch, "global_step": current_global_step + 1}
                with open(os.path.join(CHECKPOINT_DIR, "training_metadata.json"), 'w') as f:
                    json.dump(metadata, f)
            accelerator.wait_for_everyone() # Ensure all processes save before proceeding
            accelerator.print(f"Checkpoint saved at global step {current_global_step+1} to {CHECKPOINT_DIR}")

    train_loss = total_loss / len(train_loader)

    # -------------------------------------------------
    # VALIDATION
    # -------------------------------------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            outputs = model(batch)
            loss, components = improved_numeric_loss(
                                anchor_emb   = outputs["a_emb"],
                                pos_emb      = outputs["p_emb"],
                                neg_emb      = outputs["n_emb"],
                                anchor_score = outputs["a_score"],
                                pos_score    = outputs["p_score"],
                                neg_score    = outputs["n_score"],
                                anchor_value = batch["anchor_number"],
                                pos_value    = batch["positive_number"],
                                neg_value    = batch["negative_number"],
                                alpha        = 0.5,
                                beta         = 0.4,
                            )
            val_loss += loss.item()

    val_loss /= len(val_loader)

    wandb.log({
         "epoch": epoch + 1,
         "train_loss": train_loss,
         "val_loss": val_loss
     })

    accelerator.print(
        f"Epoch {epoch+1} | Train: {train_loss:.4f} | Val: {val_loss:.4f}"
    )

    # -----------------------------------------------------
    # SAVE MODEL AND TOKENIZER AFTER EACH EPOCH (for final model export)
    # -----------------------------------------------------
    accelerator.wait_for_everyone()
    unwrapped = accelerator.unwrap_model(model)
    save_path = os.path.join(WORK_DIR, f"trained_model/modernbert_lora_contrastive_corrected_CLD_r_epoch_{epoch+1}")
    os.makedirs(save_path, exist_ok=True)
    unwrapped.encoder.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    accelerator.print(f"Model and tokenizer saved for epoch {epoch+1} to {save_path}")

# -----------------------------------------------------
# WANDB FINISH (after all epochs complete)
# -----------------------------------------------------
wandb.finish()

Epoch 4:   0%|          | 0/2596 [00:37<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name)


In [ ]:
!ls

sample_data


In [ ]:
accelerator.wait_for_everyone()
unwrapped = accelerator.unwrap_model(model)
unwrapped.encoder.save_pretrained("modernbert_lora_contrastive_corrected_dynamic")
tokenizer.save_pretrained("modernbert_lora_contrastive_corrected_dynamic")

wandb.finish()

In [ ]:
epoch=0
accelerator.wait_for_everyone()
unwrapped = accelerator.unwrap_model(model)
save_path = os.path.join(WORK_DIR, f"trained_model/modernbert_lora_contrastive_corrected_ML_epoch_{epoch+1}")
os.makedirs(save_path, exist_ok=True)
unwrapped.encoder.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
accelerator.print(f"Model and tokenizer saved for epoch {epoch+1} to {save_path}")

Model and tokenizer saved for epoch 1 to /content/drive/MyDrive/numeric_finetune_data/trained_model/modernbert_lora_contrastive_corrected_ML_epoch_1


In [ ]:
#!mkdir -p drive/MyDrive/numeric_finetune_data/trained_model


In [ ]:
! cp -r modernbert_lora_contrastive-corrected2 drive/MyDrive/numeric_finetune_data/trained_model/